In [ ]:
import time as time_module
from pathlib import Path
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings("ignore", category=TqdmWarning)
import urllib3
urllib3.disable_warnings()
import pandas as pd
from IPython.display import display

from utils import (
    compute_gap_dtw_metrics,
    compute_longest_trip_duration_seconds,
    create_trip_folium_map,
    generate_artificial_gap,
    load_h3_maps_from_json,
    read_holdout_images_file,
    trip_id_from_image_path,
    run_gap_filling,
 )

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
DATA_CSV = DATA_DIR / "augmented_data.csv"
COLOR_MAP_JSON = DATA_DIR / "h3_color_map.json"
POSITION_MAP_JSON = DATA_DIR / "h3_position_map.json"
HOLDOUT_FILE = PROJECT_ROOT / "holdout_images.txt"
MODEL_PATH = PROJECT_ROOT / "h3_rgb_unet_v2.pth"
OUTPUT_DIR = PROJECT_ROOT / "inference_results"

# Set to None to keep the default rename behavior from utils.py.
custom_rename_map = None
# Example:
# custom_rename_map = {
#     "LON": "lon",
#     "LAT": "lat",
#     "# Timestamp": "time",
#     "h3_cell_9": "h3",
#     "TRIP": "trip_id",
# }

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Inference 

### Gap Filling Functions

The reconstruction logic is split into three functions that are used together in the execution cell below:

**`fill_small_gaps_interpolation()`** (Step 1): Identifies contiguous missing segments in the trajectory and fills those shorter than a configurable threshold, default 2-4 minutes, using linear interpolation between known boundary points. For each filled point, lon/lat are interpolated proportionally based on elapsed time, and the H3 cell is computed via `h3.latlng_to_cell()`. This step is computationally inexpensive and handles brief GPS dropouts with high accuracy. Returns the partially filled DataFrame and a list of remaining large-gap segments.

**`run_inpainting_inference()`** (Step 2): Converts the current trajectory state, with small gaps already filled, into a wave map image, marks remaining large gaps as white masked regions, loads the pre-trained U-Net model, and runs inference. The model output is quantized to valid H3 colors using the k-d tree quantizer. Returns the inpainted image array for coordinate extraction in Step 3.

**`fill_large_gaps_from_inpainted_image()`** (Step 3): For each large-gap segment, samples the inpainted image at regular intervals, default every 2 minutes. At each sample time, it extracts the median H3 cell position from non-white pixels in the corresponding image row. These inpainted anchor points are then combined with known boundary positions to interpolate filled coordinates for every missing timestamp in the gap.

**`run_gap_filling()`**: A wrapper that executes all three stages, produces summary metrics, and can optionally save the filled trajectory to CSV. It can also generate diagnostic visualizations comparing the original, inpainted, and masked images.

In [ ]:
trips_augmented = pd.read_csv(DATA_CSV)
if custom_rename_map:
    preview_rename_map = {
        key: value
        for key, value in custom_rename_map.items()
        if key in trips_augmented.columns
    }
    if preview_rename_map:
        trips_augmented = trips_augmented.rename(columns=preview_rename_map)
trips_augmented['time'] = pd.to_datetime(trips_augmented['time'], errors='coerce')
h3_color_map, h3_position_map = load_h3_maps_from_json(COLOR_MAP_JSON, POSITION_MAP_JSON)
longest_duration = compute_longest_trip_duration_seconds(
    trips_augmented,
    rename_map=custom_rename_map,
 )
holdout_image_paths = read_holdout_images_file(HOLDOUT_FILE)
holdout_trip_ids = [trip_id_from_image_path(path) for path in holdout_image_paths]

print(f"Loaded {len(trips_augmented):,} augmented rows")
print(f"Longest trip duration: {longest_duration:.0f} seconds")
print(f"Loaded {len(holdout_trip_ids)} holdout trips from {HOLDOUT_FILE}")
holdout_trip_ids[:10]

### Execution: Holdout Evaluation with Artificial Gaps

Run the notebook from top to bottom:

1. Execute the setup cell to load paths, model location, and optional column renaming.
2. Execute the data-loading cell to read `augmented_data.csv`, load the H3 maps, and collect the holdout trip IDs.
3. Execute the evaluation cell below to introduce an artificial large gap in each holdout trip, reconstruct the missing region, compute DTW against the ground truth, and save the filled CSV and Folium map under `inference_results/`.
4. Execute the summary cell to inspect aggregate DTW, H3 match rate, fill rate, runtime, and output file paths.

In [ ]:
results_list = []

for index, test_trip_id in enumerate(holdout_trip_ids, start=1):
    print(f"\n{'=' * 70}")
    print(f"[{index}/{len(holdout_trip_ids)}] Processing trip: {test_trip_id}")
    print(f"{'=' * 70}")

    test_trip = trips_augmented[trips_augmented['trip_id'] == test_trip_id].copy()
    if len(test_trip) == 0:
        print(f"  WARNING: No data found for trip_id={test_trip_id}, skipping.")
        results_list.append({
            'trip_id': test_trip_id,
            'n_rows': 0,
            'gap_size': 0,
            'dtw_lon': float('nan'),
            'dtw_lat': float('nan'),
            'dtw_combined': float('nan'),
            'h3_match_pct': float('nan'),
            'fill_rate': float('nan'),
            'elapsed_seconds': float('nan'),
            'filled_trip_csv': None,
            'map_file': None,
            'status': 'no_data',
        })
        continue

    test_trip = test_trip.sort_values('time').reset_index(drop=True)
    test_trip_with_missing, gap_metadata = generate_artificial_gap(
        test_trip,
        gap_start_fraction=0.75,
        gap_end_fraction=0.95,
        rename_map=custom_rename_map,
    )
    gap_start = gap_metadata['gap_start_index']
    gap_end = gap_metadata['gap_end_index']
    gap_size = gap_metadata['gap_size']

    print(f"Trip has {len(test_trip)} rows")
    print(f"Created large gap: indices {gap_start}-{gap_end} ({gap_size} rows)")

    try:
        t_start = time_module.time()
        filled_trip, metrics = run_gap_filling(
            trip_df=test_trip_with_missing,
            trip_id=test_trip_id,
            h3_color_map=h3_color_map,
            h3_position_map=h3_position_map,
            model_path=MODEL_PATH,
            longest_trip_duration_seconds=longest_duration,
            h3_resolution=9,
            small_gap_threshold_seconds=120,
            large_gap_sample_interval_seconds=120,
            n_bins=128,
            output_csv=None,
            verbose=False,
            plot=False,
            rename_map=custom_rename_map,
        )
        elapsed_seconds = time_module.time() - t_start

        filled_trip_out = filled_trip.copy()
        filled_trip_out['was_missing'] = test_trip_with_missing['lon'].isna().to_numpy()
        trip_csv_path = OUTPUT_DIR / f"filled_trip_{test_trip_id}.csv"
        filled_trip_out.to_csv(trip_csv_path, index=False)

        gap_metrics = compute_gap_dtw_metrics(
            test_trip.iloc[gap_start:gap_end],
            filled_trip.iloc[gap_start:gap_end],
        )
        map_path = create_trip_folium_map(
            original_trip_df=test_trip,
            filled_trip_df=filled_trip,
            gap_start_index=gap_start,
            gap_end_index=gap_end,
            output_path=OUTPUT_DIR / f"trip_map_{test_trip_id}.html",
            trip_id=test_trip_id,
        )

        results_list.append({
            'trip_id': test_trip_id,
            'n_rows': len(test_trip),
            'gap_size': gap_metrics['gap_size'],
            'dtw_lon': gap_metrics['dtw_lon'],
            'dtw_lat': gap_metrics['dtw_lat'],
            'dtw_combined': gap_metrics['dtw_combined'],
            'h3_match_count': gap_metrics['h3_match_count'],
            'h3_match_pct': gap_metrics['h3_match_pct'],
            'fill_rate': metrics.get('fill_rate', float('nan')),
            'elapsed_seconds': elapsed_seconds,
            'filled_trip_csv': trip_csv_path.as_posix(),
            'map_file': map_path,
            'status': 'success',
        })

        print(f"Execution completed in {elapsed_seconds:.1f}s")
        print(f"DTW (lon):      {gap_metrics['dtw_lon']:.6f}")
        print(f"DTW (lat):      {gap_metrics['dtw_lat']:.6f}")
        print(f"DTW (combined): {gap_metrics['dtw_combined']:.6f}")
        print(
            f"H3 match:       {gap_metrics['h3_match_count']}/{gap_metrics['gap_size']} "
            f"({gap_metrics['h3_match_pct']:.1f}%)"
        )
        print(f"Saved filled trip CSV to {trip_csv_path}")
        print(f"Saved folium map to {map_path}")
    except Exception as exc:
        print(f"ERROR: {exc}")
        results_list.append({
            'trip_id': test_trip_id,
            'n_rows': len(test_trip),
            'gap_size': gap_size,
            'dtw_lon': float('nan'),
            'dtw_lat': float('nan'),
            'dtw_combined': float('nan'),
            'h3_match_count': 0,
            'h3_match_pct': float('nan'),
            'fill_rate': float('nan'),
            'elapsed_seconds': float('nan'),
            'filled_trip_csv': None,
            'map_file': None,
            'status': f'error: {exc}',
        })

dtw_results_df = pd.DataFrame(results_list)
results_csv = OUTPUT_DIR / 'holdout_trips_dtw_results.csv'
dtw_results_df.to_csv(results_csv, index=False)
print(f"\nResults saved to: {results_csv}")

### DTW Evaluation Summary

The final table aggregates DTW, H3 match rate, fill rate, runtime, and output file paths for every holdout trip processed by the execution cell above.

In [ ]:
successful = dtw_results_df[dtw_results_df['status'] == 'success']
print(f"Trips evaluated successfully: {len(successful)} / {len(dtw_results_df)}")
print(f"Summary CSV: {results_csv}")

if len(successful) > 0:
    print(f"\n{'Metric':<20} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12}")
    for column in ['dtw_lon', 'dtw_lat', 'dtw_combined', 'h3_match_pct', 'fill_rate', 'elapsed_seconds']:
        values = successful[column].dropna()
        if len(values) > 0:
            print(
                f"{column:<20} {values.mean():>12.4f} {values.std():>12.4f} "
                f"{values.min():>12.4f} {values.max():>12.4f}"
            )

display(dtw_results_df)